# BLS OEWS + O*NET Merge
Stack multi-year nat3d files, then join with O*NET AI exposure data.
Includes: OCC_GROUP fix, SOC crosswalk, OCC_TITLE dedup.

In [1]:
import pandas as pd
import glob
import os
import re

# ------------------------------------------------------------------
# STEP 1: Stack all yearly nat3d files into one panel
# Handles both .xlsx (2015+) and .xls (2013-2014, pre-2015 BLS format)
# ------------------------------------------------------------------
files = glob.glob('*.xlsx') + glob.glob('*.xls')
files = [f for f in files if 'nat3d' in f.lower()]
print(f'Found {len(files)} nat3d files:', sorted(files))

dfs = []
for f in sorted(files):
    # Extract 4-digit year from filename regardless of naming convention
    match = re.search(r'(20\d{2})', f)
    if not match:
        print(f'Skipping (could not extract year): {f}')
        continue
    year = int(match.group(1))

    # pandas auto-selects engine by extension (.xls -> xlrd, .xlsx -> openpyxl)
    # but wrap in try/except since old .xls files sometimes need explicit engine
    try:
        df = pd.read_excel(f)
    except Exception as e:
        print(f'  Retry with explicit xlrd engine for {f}: {e}')
        df = pd.read_excel(f, engine='xlrd')

    df.columns = df.columns.str.upper()  # Standardise column names
    df['year'] = year
    dfs.append(df)
    print(f'  Loaded {f} → year={year}, rows={len(df)}, cols={len(df.columns)}')

panel = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows: {len(panel)}')
print(f'Years: {sorted(panel["year"].unique())}')

Found 14 nat3d files: ['nat3d_M2012_dl.xls', 'nat3d_M2013_dl.xls', 'nat3d_M2014_dl.xlsx', 'nat3d_M2015_dl.xlsx', 'nat3d_M2016_dl.xlsx', 'nat3d_M2017_dl.xlsx', 'nat3d_M2018_dl.xlsx', 'nat3d_M2019_dl.xlsx', 'nat3d_M2020_dl.xlsx', 'nat3d_M2021_dl.xlsx', 'nat3d_M2022_dl.xlsx', 'nat3d_M2023_dl.xlsx', 'nat3d_M2024_dl.xlsx', 'nat3d_M2025_dl.xlsx']
  Loaded nat3d_M2012_dl.xls → year=2012, rows=38324, cols=25
  Loaded nat3d_M2013_dl.xls → year=2013, rows=38297, cols=25
  Loaded nat3d_M2014_dl.xlsx → year=2014, rows=38570, cols=25
  Loaded nat3d_M2015_dl.xlsx → year=2015, rows=38781, cols=25
  Loaded nat3d_M2016_dl.xlsx → year=2016, rows=39047, cols=25
  Loaded nat3d_M2017_dl.xlsx → year=2017, rows=39020, cols=25
  Loaded nat3d_M2018_dl.xlsx → year=2018, rows=39380, cols=25
  Loaded nat3d_M2019_dl.xlsx → year=2019, rows=37957, cols=31
  Loaded nat3d_M2020_dl.xlsx → year=2020, rows=37714, cols=32
  Loaded nat3d_M2021_dl.xlsx → year=2021, rows=38720, cols=33
  Loaded nat3d_M2022_dl.xlsx → year=202

In [2]:
# ------------------------------------------------------------------
# STEP 2: Keep only detailed-level occupations
# Handles column name change: OCC_GROUP (2015-2018) vs O_GROUP (2019+)
# ------------------------------------------------------------------
if 'OCC_GROUP' in panel.columns and 'O_GROUP' in panel.columns:
    panel['OCC_GROUP'] = panel['OCC_GROUP'].fillna(panel['O_GROUP'])
elif 'O_GROUP' in panel.columns:
    panel['OCC_GROUP'] = panel['O_GROUP']

panel = panel[panel['OCC_GROUP'] == 'detailed'].copy()
print(f'Rows after keeping detailed only: {len(panel)}')
print(f'Years: {sorted(panel["year"].unique())}')
print(f'Unique OCC_CODEs: {panel["OCC_CODE"].nunique()}')
print(f'Unique NAICS: {panel["NAICS"].nunique()}')

Rows after keeping detailed only: 257227
Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Unique OCC_CODEs: 965
Unique NAICS: 96


In [3]:
# ------------------------------------------------------------------
# STEP 3: Apply SOC 2010 -> 2018 crosswalk to 2015-2018 rows
# Download crosswalk from:
# https://www.bls.gov/soc/2018/soc_2010_to_2018_crosswalk.xlsx
# ------------------------------------------------------------------
xw_raw = pd.read_excel('soc_2010_to_2018_crosswalk.xlsx', skiprows=7)
xw_raw.columns = ['SOC_2010', 'TITLE_2010', 'SOC_2018', 'TITLE_2018']
xw_raw = xw_raw.dropna(subset=['SOC_2010', 'SOC_2018'])
xw_raw['SOC_2010'] = xw_raw['SOC_2010'].astype(str).str.strip().str.replace('#', '', regex=False)
xw_raw['SOC_2018'] = xw_raw['SOC_2018'].astype(str).str.strip().str.replace('#', '', regex=False)

# For split cases: prefer same code, otherwise take first mapping
def pick_best_mapping(group):
    same_code = group[group['SOC_2010'] == group['SOC_2018']]
    if len(same_code) > 0:
        return same_code.iloc[0]
    return group.iloc[0]

xw_clean = xw_raw.groupby('SOC_2010', group_keys=False).apply(pick_best_mapping)
xw_clean = xw_clean[['SOC_2010', 'SOC_2018']].reset_index(drop=True)
print(f'Crosswalk rows (1-to-1): {len(xw_clean)}')

# Apply crosswalk to pre-2019 rows only
pre  = panel[panel['year'] <= 2018].copy()
post = panel[panel['year'] >= 2019].copy()

pre = pre.merge(xw_clean, left_on='OCC_CODE', right_on='SOC_2010', how='left')
unmapped = pre['SOC_2018'].isna().sum()
print(f'Pre-2019 rows unmapped: {unmapped} / {len(pre)}')

pre['OCC_CODE'] = pre['SOC_2018'].fillna(pre['OCC_CODE'])
pre = pre.drop(columns=['SOC_2010', 'SOC_2018'])

panel = pd.concat([pre, post], ignore_index=True)
print(f'Panel after crosswalk: {len(panel)} rows')

C:\Users\asus\AppData\Local\Temp\ipykernel_7632\286098827.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  xw_clean = xw_raw.groupby('SOC_2010', group_keys=False).apply(pick_best_mapping)


Crosswalk rows (1-to-1): 841
Pre-2019 rows unmapped: 853 / 128112
Panel after crosswalk: 257227 rows


In [4]:
# ------------------------------------------------------------------
# STEP 3.5: Diagnose the duplicate occ-naics-year keys
# ------------------------------------------------------------------
dup_keys = (
    panel.groupby(['OCC_CODE', 'NAICS', 'year'])
    .size()
    .reset_index(name='n')
)
dup_keys = dup_keys[dup_keys['n'] > 1]
print(f'Number of duplicated occ-naics-year keys: {len(dup_keys)}')
print(f'Affected OCC_CODEs: {sorted(dup_keys["OCC_CODE"].unique())}')

# Cross-check against the crosswalk: which SOC_2010 codes collapse onto
# the same SOC_2018 code? (many-to-one merges are the likely culprit)
many_to_one = (
    xw_clean.groupby('SOC_2018')['SOC_2010']
    .nunique()
    .reset_index(name='n_sources')
)
many_to_one = many_to_one[many_to_one['n_sources'] > 1]
print(f'\nSOC_2018 codes receiving >1 SOC_2010 source code: {len(many_to_one)}')
print(many_to_one.merge(
    xw_clean[xw_clean['SOC_2018'].isin(many_to_one['SOC_2018'])],
    on='SOC_2018'
).sort_values('SOC_2018').head(20))

Number of duplicated occ-naics-year keys: 990
Affected OCC_CODEs: ['15-1252', '15-2099', '25-4022', '27-3023', '27-3099', '29-2099', '35-3023', '39-1013', '43-2099', '47-5044', '47-5049', '49-9099', '51-9124', '53-4022']

SOC_2018 codes receiving >1 SOC_2010 source code: 16
   SOC_2018  n_sources SOC_2010
0   15-1252          2  15-1132
1   15-1252          2  15-1133
2   15-2099          2  15-2091
3   15-2099          2  15-2099
4   17-3029          2  17-3029
5   17-3029          2  55-3017
6   25-4022          2  25-4021
7   25-4022          2  25-9011
8   27-3023          2  27-3021
9   27-3023          2  27-3022
10  27-3099          2  27-3012
11  27-3099          2  27-3099
12  29-2099          2  29-2054
13  29-2099          2  29-2099
15  35-3023          2  35-3022
14  35-3023          2  35-3021
16  39-1013          2  39-1011
17  39-1013          2  39-1012
18  43-2099          2  27-4013
19  43-2099          2  43-2099


In [5]:
# ------------------------------------------------------------------
# STEP 4: Collapse duplicate occ-naics-year keys created by the
# many-to-one SOC crosswalk, then keep only the columns we need
# ------------------------------------------------------------------
panel['TOT_EMP']  = pd.to_numeric(panel['TOT_EMP'], errors='coerce')
panel['H_MEDIAN'] = pd.to_numeric(panel['H_MEDIAN'], errors='coerce')
panel['A_MEDIAN'] = pd.to_numeric(panel['A_MEDIAN'], errors='coerce')

def collapse_group(g):
    tot_emp = g['TOT_EMP'].sum(min_count=1)  # min_count=1 -> stays NaN if all NaN, not 0
    # Employment-weighted average wage as the merge approximation for two
    # medians being combined into one occupation code. Note: this is an
    # approximation (weighted mean of medians is not itself a median),
    # should be flagged as a methodological note in the data appendix.
    w = g['TOT_EMP']
    if w.notna().sum() > 0 and w.sum() > 0:
        h_wage = np.average(g['H_MEDIAN'].fillna(g['H_MEDIAN'].mean()), weights=w.fillna(0) + 1e-9)
        a_wage = np.average(g['A_MEDIAN'].fillna(g['A_MEDIAN'].mean()), weights=w.fillna(0) + 1e-9)
    else:
        h_wage = g['H_MEDIAN'].mean()
        a_wage = g['A_MEDIAN'].mean()
    return pd.Series({
        'NAICS_TITLE': g['NAICS_TITLE'].iloc[0],
        'OCC_TITLE':   g['OCC_TITLE'].iloc[0],
        'TOT_EMP':     tot_emp,
        'H_MEDIAN':    h_wage,
        'A_MEDIAN':    a_wage,
    })

import numpy as np

before = len(panel)
panel = (
    panel.groupby(['year', 'NAICS', 'OCC_CODE'], group_keys=False)
    .apply(collapse_group)
    .reset_index()
)
print(f'Rows before collapse: {before}')
print(f'Rows after collapse: {len(panel)}')

dups = panel.groupby(['OCC_CODE', 'NAICS', 'year']).size().max()
print(f'Max rows per occ-naics-year (should be 1 now): {dups}')

Rows before collapse: 257227
Rows after collapse: 256237
Max rows per occ-naics-year (should be 1 now): 1


C:\Users\asus\AppData\Local\Temp\ipykernel_7632\870442127.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_group)


In [6]:
# ------------------------------------------------------------------
# STEP 5: Load O*NET merged task descriptions
# ------------------------------------------------------------------
onet = pd.read_csv('onet_merged_tasks.csv')

# O*NET SOC format is '11-1011.00', BLS format is '11-1011'
onet['OCC_CODE'] = onet['O*NET-SOC Code'].str[:7]
onet = onet.drop_duplicates(subset='OCC_CODE')
print(f'O*NET occupations after dedup: {len(onet)}')

O*NET occupations after dedup: 798


In [7]:
# ------------------------------------------------------------------
# STEP 6: Merge panel with O*NET
# ------------------------------------------------------------------
merged = panel.merge(onet[['OCC_CODE', 'title', 'task_count', 'combined_tasks']],
                     on='OCC_CODE',
                     how='left')

matched = merged['combined_tasks'].notna().sum()
total = len(merged)
print(f'Rows with O*NET task description: {matched} / {total} ({matched/total*100:.1f}%)')

Rows with O*NET task description: 236411 / 256237 (92.3%)


In [8]:
# ------------------------------------------------------------------
# STEP 7: Merge with Felten AIOE index
# ------------------------------------------------------------------
felten = pd.read_excel('Language_Modeling_AIOE_and_AIIE.xlsx')

# Rename columns to match the rest of the pipeline
felten = felten.rename(columns={
    'SOC Code':               'OCC_CODE',
    'Language Modeling AIOE': 'LM_AIOE'
})

felten['OCC_CODE'] = felten['OCC_CODE'].astype(str).str.strip()
felten = felten.drop_duplicates(subset='OCC_CODE')
print(f'Felten occupations: {len(felten)}')

merged = merged.merge(felten[['OCC_CODE', 'LM_AIOE']],
                      on='OCC_CODE', how='left')

aioe_match = merged['LM_AIOE'].notna().sum()
print(f'Rows with AIOE: {aioe_match} / {len(merged)} ({aioe_match/len(merged)*100:.1f}%)')
print()
print('AIOE match rate by year:')
print(merged.groupby('year')['LM_AIOE'].apply(lambda x: x.notna().mean()*100).round(1))

Felten occupations: 774
Rows with AIOE: 208020 / 256237 (81.2%)

AIOE match rate by year:
year
2012    83.0
2013    83.1
2014    83.0
2015    83.2
2016    83.4
2017    82.6
2018    82.3
2019    80.3
2020    80.0
2021    79.3
2022    79.2
2023    79.1
2024    79.1
2025    79.1
Name: LM_AIOE, dtype: float64


In [9]:
# ------------------------------------------------------------------
# STEP 8: Merge Microsoft AI Applicability Score
# ------------------------------------------------------------------
ms = pd.read_csv('ai_applicability_scores.csv')
ms = ms.rename(columns={
    'SOC Code':              'OCC_CODE',
    'ai_applicability_score': 'MS_SCORE'
})
ms['OCC_CODE'] = ms['OCC_CODE'].astype(str).str.strip()
ms = ms.drop_duplicates(subset='OCC_CODE')

merged = merged.merge(ms[['OCC_CODE', 'MS_SCORE']],
                      on='OCC_CODE', how='left')

ms_match = merged['MS_SCORE'].notna().sum()
print(f'Rows with MS_SCORE: {ms_match} / {len(merged)} ({ms_match/len(merged)*100:.1f}%)')

merged.to_csv('bls_onet_felten_ms_panel_harmonised.csv', index=False)
print(f'Saved to bls_onet_felten_ms_panel_harmonised.csv — {len(merged)} rows')

Rows with MS_SCORE: 236627 / 256237 (92.3%)
Saved to bls_onet_felten_ms_panel_harmonised.csv — 256237 rows
